## IMPORTS

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata

from pathlib import Path
from IPython.display import display

## PROJECT PATHS

In [12]:
PROJECT_ROOT = Path(
    r"C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor"
)

DATA_DIR = PROJECT_ROOT / "Data"
RAW_DIR = DATA_DIR / "Raw"
CLEAN_DIR = DATA_DIR / "Clean"
AUDIT_DIR = DATA_DIR / "Audit"

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Dataset A
DATASET_A_PATH = CLEAN_DIR / "cleaned_movies.csv"

# Dataset B
DATASET_B_PATH = RAW_DIR / "archive" / "movies.csv"

# Dataset C
DATASET_C_PATH = RAW_DIR / "21920642" / "movies_raw.csv"

print("Dataset A:", DATASET_A_PATH)
print("Dataset B:", DATASET_B_PATH)
print("Dataset C:", DATASET_C_PATH)

Dataset A: C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Clean\cleaned_movies.csv
Dataset B: C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Raw\archive\movies.csv
Dataset C: C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Raw\21920642\movies_raw.csv


## LOAD DATASETS

In [ ]:
#Load source datasets
import csv
import sys

# Allow very large CSV fields
csv.field_size_limit(sys.maxsize)


#Dataset A
cleaned_movies = pd.read_csv(
    DATASET_A_PATH,
    encoding="utf-8"
)


#Dataset B

tmdb_relational = pd.read_csv(
    DATASET_B_PATH,
    engine="python",
    encoding="utf-8"
)


#Dataset C

tmdb_recent = pd.read_csv(
    DATASET_C_PATH,
    encoding="utf-8"
)


#Load Confirmation

print("=" * 80)
print("SOURCE DATASETS LOADED")
print("=" * 80)

print(
    f"Dataset A - Cleaned Movies: "
    f"{cleaned_movies.shape}"
)

print(
    f"Dataset B - TMDB Relational: "
    f"{tmdb_relational.shape}"
)

print(
    f"Dataset C - TMDB Recent: "
    f"{tmdb_recent.shape}"
)

SOURCE DATASETS LOADED
Dataset A - Cleaned Movies: (7668, 20)
Dataset B - TMDB Relational: (9771, 22)
Dataset C - TMDB Recent: (17978, 16)


## GENERAL HELPER FUNCTIONS

In [15]:
def find_column(df, candidates):
    
    #Return the first candidate column that exists
    #in the dataframe.

    for column in candidates:

        if column in df.columns:
            return column

    return None


def percentage(
    count,
    total
):
    """
    Safely calculate a percentage.
    """

    if total == 0:
        return 0.0

    return (
        count / total
    ) * 100

## BASIC DATASET AUDIT FUNCTION

In [16]:
def basic_dataset_audit(
    df,
    dataset_name
):
    #Display basic structural information
    #about a dataframe.

    print("\n" + "=" * 80)
    print(dataset_name.upper())
    print("=" * 80)


    # --------------------------------------------------------
    # SHAPE
    # --------------------------------------------------------

    print("\nSHAPE")
    print("-" * 40)

    print(
        f"Rows:    {len(df):,}"
    )

    print(
        f"Columns: {len(df.columns):,}"
    )


   # Column Names

    print("\nCOLUMN NAMES")
    print("-" * 40)

    for column in df.columns:
        print(column)


  #Duplicate Rows

    duplicate_rows = (
        df.duplicated().sum()
    )

    print("\nDUPLICATE ROWS")
    print("-" * 40)

    print(
        f"Duplicate rows: "
        f"{duplicate_rows:,}"
    )


   #Missing Values

    missing_summary = pd.DataFrame({
        "missing_count":
            df.isna().sum(),

        "missing_percent":
            (
                df.isna().mean()
                * 100
            ).round(2)
    })

    missing_summary = (
        missing_summary[
            missing_summary[
                "missing_count"
            ] > 0
        ]
        .sort_values(
            "missing_count",
            ascending=False
        )
    )

    print("\nMISSING VALUES")
    print("-" * 40)

    if missing_summary.empty:

        print(
            "No missing values detected."
        )

    else:

        display(
            missing_summary
        )

## RUN BASIC AUDITS

In [17]:
basic_dataset_audit(
    cleaned_movies,
    "Dataset A - Cleaned Industry Dataset"
)

basic_dataset_audit(
    tmdb_relational,
    "Dataset B - TMDB Relational Dataset"
)

basic_dataset_audit(
    tmdb_recent,
    "Dataset C - TMDB Recent Dataset"
)


DATASET A - CLEANED INDUSTRY DATASET

SHAPE
----------------------------------------
Rows:    7,668
Columns: 20

COLUMN NAMES
----------------------------------------
name
rating
genre
year
released
score
votes
director
writer
star
country
budget
gross
company
runtime
Profit
ROI
released_month
release_quarter
decade

DUPLICATE ROWS
----------------------------------------
Duplicate rows: 0

MISSING VALUES
----------------------------------------


,missing_count,missing_percent
ROI,2232,29.11
Profit,2232,29.11
budget,2171,28.31
gross,189,2.46
rating,77,1.00
released,59,0.77
released_month,59,0.77
release_quarter,59,0.77
company,17,0.22
runtime,4,0.05



DATASET B - TMDB RELATIONAL DATASET

SHAPE
----------------------------------------
Rows:    9,771
Columns: 22

COLUMN NAMES
----------------------------------------
id
title
original_title
overview
release_date
runtime
budget
revenue
vote_average
vote_count
popularity
poster_path
backdrop_path
status
tagline
homepage
original_language
adult
video
created_at
updated_at
genres

DUPLICATE ROWS
----------------------------------------
Duplicate rows: 0

MISSING VALUES
----------------------------------------


,missing_count,missing_percent
homepage,6337,64.86
tagline,3803,38.92
backdrop_path,1519,15.55
poster_path,338,3.46
genres,270,2.76
overview,160,1.64
release_date,61,0.62
updated_at,2,0.02
created_at,2,0.02
runtime,1,0.01



DATASET C - TMDB RECENT DATASET

SHAPE
----------------------------------------
Rows:    17,978
Columns: 16

COLUMN NAMES
----------------------------------------
movie_id
title
release_date
status
budget
revenue
runtime
genres
production_companies
production_countries
spoken_languages
original_language
vote_average
vote_count
popularity
adult

DUPLICATE ROWS
----------------------------------------
Duplicate rows: 0

MISSING VALUES
----------------------------------------


,missing_count,missing_percent
production_companies,494,2.75
production_countries,176,0.98
spoken_languages,92,0.51
genres,10,0.06


## IDENTIFY IMPORTANT COLUMNS

In [20]:
dataset_columns = {
    "A": {
        "title": find_column(
            cleaned_movies,
            ["name", "title"]
        ),
        "budget": find_column(
            cleaned_movies,
            ["budget"]
        ),
        "revenue": find_column(
            cleaned_movies,
            ["gross", "revenue"]
        ),
        "year": find_column(
            cleaned_movies,
            ["year", "release_year"]
        ),
        "release_date": find_column(
            cleaned_movies,
            ["released", "release_date"]
        )
    },

    "B": {
        "title": find_column(
            tmdb_relational,
            ["title", "name"]
        ),
        "budget": find_column(
            tmdb_relational,
            ["budget"]
        ),
        "revenue": find_column(
            tmdb_relational,
            ["revenue", "gross"]
        ),
        "year": find_column(
            tmdb_relational,
            ["year", "release_year"]
        ),
        "release_date": find_column(
            tmdb_relational,
            ["release_date", "released"]
        )
    },

    "C": {
        "title": find_column(
            tmdb_recent,
            ["title", "name"]
        ),
        "budget": find_column(
            tmdb_recent,
            ["budget"]
        ),
        "revenue": find_column(
            tmdb_recent,
            ["revenue", "gross"]
        ),
        "year": find_column(
            tmdb_recent,
            ["year", "release_year"]
        ),
        "release_date": find_column(
            tmdb_recent,
            ["release_date", "released"]
        )
    }
}

print("=" * 80)
print("IDENTIFIED IMPORTANT COLUMNS")
print("=" * 80)

for dataset, columns in dataset_columns.items():
    print(f"\nDataset {dataset}")
    print("-" * 40)

    for column_type, column_name in columns.items():
        print(
            f"{column_type:15}: "
            f"{column_name}"
        )

IDENTIFIED IMPORTANT COLUMNS

Dataset A
----------------------------------------
title          : name
budget         : budget
revenue        : gross
year           : year
release_date   : released

Dataset B
----------------------------------------
title          : title
budget         : budget
revenue        : revenue
year           : None
release_date   : release_date

Dataset C
----------------------------------------
title          : title
budget         : budget
revenue        : revenue
year           : None
release_date   : release_date


## NUMERIC QUALITY HELPER

In [21]:
def numeric_quality_report(
    df,
    column,
    dataset_name
):
    #Audit a numeric column for missing,
    #zero, positive and negative values.

    print(
        f"\n{dataset_name}"
    )

    print("-" * 60)


    if column is None:

        print(
            "Column not found."
        )

        return


    values = pd.to_numeric(
        df[column],
        errors="coerce"
    )


    total = len(values)

    missing = (
        values.isna().sum()
    )

    zero = (
        values.eq(0).sum()
    )

    positive = (
        values.gt(0).sum()
    )

    negative = (
        values.lt(0).sum()
    )


    print(
        f"Column:                "
        f"{column}"
    )

    print(
        f"Total rows:            "
        f"{total:,}"
    )

    print(
        f"Missing values:        "
        f"{missing:,} "
        f"({percentage(missing, total):.2f}%)"
    )

    print(
        f"Zero values:           "
        f"{zero:,} "
        f"({percentage(zero, total):.2f}%)"
    )

    print(
        f"Positive values:       "
        f"{positive:,} "
        f"({percentage(positive, total):.2f}%)"
    )

    print(
        f"Negative values:       "
        f"{negative:,}"
    )

## BUDGET QUALITY AUDIT

In [22]:
print("=" * 80)
print("BUDGET DATA QUALITY")
print("=" * 80)


numeric_quality_report(
    cleaned_movies,
    dataset_columns["A"]["budget"],
    "Dataset A - Cleaned Movies"
)


numeric_quality_report(
    tmdb_relational,
    dataset_columns["B"]["budget"],
    "Dataset B - TMDB Relational Movies"
)


numeric_quality_report(
    tmdb_recent,
    dataset_columns["C"]["budget"],
    "Dataset C - TMDB Recent Movies"
)

BUDGET DATA QUALITY

Dataset A - Cleaned Movies
------------------------------------------------------------
Column:                budget
Total rows:            7,668
Missing values:        2,171 (28.31%)
Zero values:           0 (0.00%)
Positive values:       5,497 (71.69%)
Negative values:       0

Dataset B - TMDB Relational Movies
------------------------------------------------------------
Column:                budget
Total rows:            9,771
Missing values:        1 (0.01%)
Zero values:           5,755 (58.90%)
Positive values:       4,015 (41.09%)
Negative values:       0

Dataset C - TMDB Recent Movies
------------------------------------------------------------
Column:                budget
Total rows:            17,978
Missing values:        0 (0.00%)
Zero values:           12,463 (69.32%)
Positive values:       5,515 (30.68%)
Negative values:       0


## REVENUE QUALITY AUDIT

In [23]:
print("=" * 80)
print("REVENUE DATA QUALITY")
print("=" * 80)


numeric_quality_report(
    cleaned_movies,
    dataset_columns["A"]["revenue"],
    "Dataset A - Cleaned Movies"
)


numeric_quality_report(
    tmdb_relational,
    dataset_columns["B"]["revenue"],
    "Dataset B - TMDB Relational Movies"
)


numeric_quality_report(
    tmdb_recent,
    dataset_columns["C"]["revenue"],
    "Dataset C - TMDB Recent Movies"
)

REVENUE DATA QUALITY

Dataset A - Cleaned Movies
------------------------------------------------------------
Column:                gross
Total rows:            7,668
Missing values:        189 (2.46%)
Zero values:           0 (0.00%)
Positive values:       7,479 (97.54%)
Negative values:       0

Dataset B - TMDB Relational Movies
------------------------------------------------------------
Column:                revenue
Total rows:            9,771
Missing values:        1 (0.01%)
Zero values:           5,743 (58.78%)
Positive values:       4,027 (41.21%)
Negative values:       0

Dataset C - TMDB Recent Movies
------------------------------------------------------------
Column:                revenue
Total rows:            17,978
Missing values:        0 (0.00%)
Zero values:           11,156 (62.05%)
Positive values:       6,822 (37.95%)
Negative values:       0


## COMPLETE FINANCIAL ROWS HELPER

In [ ]:
def count_complete_financial_rows(
    df,
    budget_column,
    revenue_column
):
    #Count rows where budget and revenue
    #are both positive.

    budget = pd.to_numeric(
        df[budget_column],
        errors="coerce"
    )

    revenue = pd.to_numeric(
        df[revenue_column],
        errors="coerce"
    )

    valid = (
        budget.gt(0)
        & revenue.gt(0)
    )

    return int(
        valid.sum()
    )

## MOVIES WITH BOTH BUDGET AND REVENUE

In [25]:
financial_a = count_complete_financial_rows(
    cleaned_movies,
    dataset_columns["A"]["budget"],
    dataset_columns["A"]["revenue"]
)


financial_b = count_complete_financial_rows(
    tmdb_relational,
    dataset_columns["B"]["budget"],
    dataset_columns["B"]["revenue"]
)


financial_c = count_complete_financial_rows(
    tmdb_recent,
    dataset_columns["C"]["budget"],
    dataset_columns["C"]["revenue"]
)


print("=" * 80)
print("MOVIES WITH BOTH BUDGET AND REVENUE")
print("=" * 80)


print(
    f"Dataset A - Cleaned Movies: "
    f"{financial_a:,} movies "
    f"({percentage(financial_a, len(cleaned_movies)):.2f}%)"
)


print(
    f"Dataset B - TMDB Relational Movies: "
    f"{financial_b:,} movies "
    f"({percentage(financial_b, len(tmdb_relational)):.2f}%)"
)


print(
    f"Dataset C - TMDB Recent Movies: "
    f"{financial_c:,} movies "
    f"({percentage(financial_c, len(tmdb_recent)):.2f}%)"
)

MOVIES WITH BOTH BUDGET AND REVENUE
Dataset A - Cleaned Movies: 5,436 movies (70.89%)
Dataset B - TMDB Relational Movies: 3,435 movies (35.16%)
Dataset C - TMDB Recent Movies: 4,105 movies (22.83%)


## RELEASE YEAR HELPER

In [26]:
def extract_year_series(
    df,
    year_column=None,
    release_date_column=None
):
    #Return release year values using either
    #an explicit year column or release date.

    if year_column is not None:

        years = pd.to_numeric(
            df[year_column],
            errors="coerce"
        )

        return years


    if release_date_column is not None:

        dates = pd.to_datetime(
            df[release_date_column],
            errors="coerce"
        )

        return dates.dt.year


    return pd.Series(
        np.nan,
        index=df.index
    )


def get_year_range(
    df,
    year_column=None,
    release_date_column=None
):
    #Return minimum and maximum valid release year.

    years = extract_year_series(
        df,
        year_column,
        release_date_column
    ).dropna()


    if years.empty:
        return None, None


    return (
        int(years.min()),
        int(years.max())
    )

## RELEASE YEAR COVERAGE

In [27]:
a_start, a_end = get_year_range(
    cleaned_movies,
    dataset_columns["A"]["year"],
    dataset_columns["A"]["release_date"]
)


b_start, b_end = get_year_range(
    tmdb_relational,
    dataset_columns["B"]["year"],
    dataset_columns["B"]["release_date"]
)


c_start, c_end = get_year_range(
    tmdb_recent,
    dataset_columns["C"]["year"],
    dataset_columns["C"]["release_date"]
)


print("=" * 80)
print("RELEASE YEAR COVERAGE")
print("=" * 80)


print(
    f"Dataset A - Cleaned Movies: "
    f"{a_start} - {a_end}"
)

print(
    f"Dataset B - TMDB Relational: "
    f"{b_start} - {b_end}"
)

print(
    f"Dataset C - TMDB Recent: "
    f"{c_start} - {c_end}"
)

RELEASE YEAR COVERAGE
Dataset A - Cleaned Movies: 1980 - 2020
Dataset B - TMDB Relational: 1904 - 2028
Dataset C - TMDB Recent: 2010 - 2025


## DUPLICATE TITLE AUDIT

In [28]:
def duplicate_title_audit(
    df,
    dataset_name,
    title_column
):

    print(
        f"\n{dataset_name}"
    )

    print("-" * 60)


    if title_column is None:

        print(
            "No title column found."
        )

        return


    duplicate_rows = df[
        df.duplicated(
            subset=[title_column],
            keep=False
        )
    ]


    print(
        f"Title column:          "
        f"{title_column}"
    )

    print(
        f"Duplicate title rows: "
        f"{len(duplicate_rows):,}"
    )

    print(
        f"Duplicate titles:     "
        f"{duplicate_rows[title_column].nunique():,}"
    )

## RUN DUPLICATE TITLE AUDIT

In [29]:
print("=" * 80)
print("DUPLICATE MOVIE TITLES")
print("=" * 80)


duplicate_title_audit(
    cleaned_movies,
    "Dataset A - Cleaned Movies",
    dataset_columns["A"]["title"]
)


duplicate_title_audit(
    tmdb_relational,
    "Dataset B - TMDB Relational",
    dataset_columns["B"]["title"]
)


duplicate_title_audit(
    tmdb_recent,
    "Dataset C - TMDB Recent",
    dataset_columns["C"]["title"]
)

DUPLICATE MOVIE TITLES

Dataset A - Cleaned Movies
------------------------------------------------------------
Title column:          name
Duplicate title rows: 305
Duplicate titles:     149

Dataset B - TMDB Relational
------------------------------------------------------------
Title column:          title
Duplicate title rows: 1,099
Duplicate titles:     484

Dataset C - TMDB Recent
------------------------------------------------------------
Title column:          title
Duplicate title rows: 1,206
Duplicate titles:     550


## MOVIE ID AUDIT HELPER

In [ ]:
def movie_id_audit(
    df,
    dataset_name
):
    #Inspect movie identifier quality.

    id_column = find_column(
        df,
        [
            "movie_id",
            "tmdb_id",
            "id"
        ]
    )


    print(
        f"\n{dataset_name}"
    )

    print("-" * 60)


    if id_column is None:

        print(
            "No movie ID column found."
        )

        return


    print(
        f"ID column:       "
        f"{id_column}"
    )

    print(
        f"Rows:            "
        f"{len(df):,}"
    )

    print(
        f"Unique IDs:      "
        f"{df[id_column].nunique(dropna=True):,}"
    )

    print(
        f"Missing IDs:     "
        f"{df[id_column].isna().sum():,}"
    )

    print(
        f"Duplicate IDs:   "
        f"{df[id_column].duplicated().sum():,}"
    )

## RUN MOVIE ID AUDIT

In [31]:
print("=" * 80)
print("MOVIE ID QUALITY")
print("=" * 80)


movie_id_audit(
    cleaned_movies,
    "Dataset A"
)


movie_id_audit(
    tmdb_relational,
    "Dataset B"
)


movie_id_audit(
    tmdb_recent,
    "Dataset C"
)

MOVIE ID QUALITY

Dataset A
------------------------------------------------------------
No movie ID column found.

Dataset B
------------------------------------------------------------
ID column:       id
Rows:            9,771
Unique IDs:      9,771
Missing IDs:     0
Duplicate IDs:   0

Dataset C
------------------------------------------------------------
ID column:       movie_id
Rows:            17,978
Unique IDs:      17,978
Missing IDs:     0
Duplicate IDs:   0


## POSITIVE COUNT HELPER

In [ ]:
def count_positive_values(
    df,
    column
):
    #Count positive numeric values.

    values = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    return int(
        values.gt(0).sum()
    )

## BUILD AUDIT SUMMARY

In [33]:
audit_summary = pd.DataFrame([
    {
        "dataset":
            "Dataset A - Cleaned Movies",

        "rows":
            len(cleaned_movies),

        "columns":
            len(cleaned_movies.columns),

        "start_year":
            a_start,

        "end_year":
            a_end,

        "positive_budget":
            count_positive_values(
                cleaned_movies,
                dataset_columns["A"]["budget"]
            ),

        "positive_revenue":
            count_positive_values(
                cleaned_movies,
                dataset_columns["A"]["revenue"]
            ),

        "budget_and_revenue":
            financial_a
    },

    {
        "dataset":
            "Dataset B - TMDB Relational",

        "rows":
            len(tmdb_relational),

        "columns":
            len(tmdb_relational.columns),

        "start_year":
            b_start,

        "end_year":
            b_end,

        "positive_budget":
            count_positive_values(
                tmdb_relational,
                dataset_columns["B"]["budget"]
            ),

        "positive_revenue":
            count_positive_values(
                tmdb_relational,
                dataset_columns["B"]["revenue"]
            ),

        "budget_and_revenue":
            financial_b
    },

    {
        "dataset":
            "Dataset C - TMDB 2010-2025",

        "rows":
            len(tmdb_recent),

        "columns":
            len(tmdb_recent.columns),

        "start_year":
            c_start,

        "end_year":
            c_end,

        "positive_budget":
            count_positive_values(
                tmdb_recent,
                dataset_columns["C"]["budget"]
            ),

        "positive_revenue":
            count_positive_values(
                tmdb_recent,
                dataset_columns["C"]["revenue"]
            ),

        "budget_and_revenue":
            financial_c
    }
])


display(
    audit_summary
)

,dataset,rows,columns,start_year,end_year,positive_budget,positive_revenue,budget_and_revenue
0,Dataset A - Cleaned Movies,7668,20,1980,2020,5497,7479,5436
1,Dataset B - TMDB Relational,9771,22,1904,2028,4015,4027,3435
2,Dataset C - TMDB 2010-2025,17978,16,2010,2025,5515,6822,4105


## VALIDATE AUDIT SUMMARY

In [34]:
expected_results = {
    "Dataset A - Cleaned Movies": {
        "rows": 7668,
        "positive_budget": 5497,
        "positive_revenue": 7479,
        "budget_and_revenue": 5436
    },

    "Dataset B - TMDB Relational": {
        "rows": 9771,
        "positive_budget": 4015,
        "positive_revenue": 4027,
        "budget_and_revenue": 3435
    },

    "Dataset C - TMDB 2010-2025": {
        "rows": 17978,
        "positive_budget": 5515,
        "positive_revenue": 6822,
        "budget_and_revenue": 4105
    }
}


print("=" * 80)
print("SUMMARY VALIDATION")
print("=" * 80)


for _, row in audit_summary.iterrows():

    dataset_name = row["dataset"]

    print(
        f"\n{dataset_name}"
    )


    for metric in [
        "rows",
        "positive_budget",
        "positive_revenue",
        "budget_and_revenue"
    ]:

        actual = int(
            row[metric]
        )

        expected = (
            expected_results[
                dataset_name
            ][metric]
        )

        status = (
            "PASS"
            if actual == expected
            else "FAIL"
        )


        print(
            f"{metric}: "
            f"{actual} | "
            f"Expected: "
            f"{expected} | "
            f"{status}"
        )

SUMMARY VALIDATION

Dataset A - Cleaned Movies
rows: 7668 | Expected: 7668 | PASS
positive_budget: 5497 | Expected: 5497 | PASS
positive_revenue: 7479 | Expected: 7479 | PASS
budget_and_revenue: 5436 | Expected: 5436 | PASS

Dataset B - TMDB Relational
rows: 9771 | Expected: 9771 | PASS
positive_budget: 4015 | Expected: 4015 | PASS
positive_revenue: 4027 | Expected: 4027 | PASS
budget_and_revenue: 3435 | Expected: 3435 | PASS

Dataset C - TMDB 2010-2025
rows: 17978 | Expected: 17978 | PASS
positive_budget: 5515 | Expected: 5515 | PASS
positive_revenue: 6822 | Expected: 6822 | PASS
budget_and_revenue: 4105 | Expected: 4105 | PASS


# CROSS-DATASET MATCHING AUDIT

In [ ]:
def normalize_movie_title(
    title
):
    #Normalize movie titles for cross-dataset matching.

    if pd.isna(title):
        return pd.NA


    title = str(
        title
    ).strip().lower()


    title = unicodedata.normalize(
        "NFKD",
        title
    )


    title = "".join(
        character
        for character in title
        if not unicodedata.combining(
            character
        )
    )


    title = re.sub(
        r"[^a-z0-9\s]",
        " ",
        title
    )


    title = re.sub(
        r"\s+",
        " ",
        title
    ).strip()


    if title == "":
        return pd.NA


    return title

## PREPATE MATCHING DATASET FUNCTION

In [ ]:
def prepare_matching_dataset(
    df,
    dataset_name,
    title_column,
    year_column=None,
    release_date_column=None
):
    #Create normalized title, release year,
    #and title-year matching key.

    result = df.copy()

    # TITLE

    result["_match_title"] = (
        result[title_column]
        .apply(
            normalize_movie_title
        )
    )

    # YEAR

    years = extract_year_series(
        result,
        year_column,
        release_date_column
    )


    result["_match_year"] = (
        pd.to_numeric(
            years,
            errors="coerce"
        )
        .astype("Int64")
    )

    # MATCH KEY

    valid_match = (
        result["_match_title"].notna()
        & result["_match_year"].notna()
    )


    result["_match_key"] = pd.NA


    result.loc[
        valid_match,
        "_match_key"
    ] = (
        result.loc[
            valid_match,
            "_match_title"
        ].astype(str)
        + " | "
        + result.loc[
            valid_match,
            "_match_year"
        ].astype(str)
    )


    # SUMMARY

    print(
        f"\n{dataset_name}"
    )

    print("-" * 60)

    print(
        f"Title column:        "
        f"{title_column}"
    )

    print(
        f"Year column:         "
        f"{year_column}"
    )

    print(
        f"Release date column: "
        f"{release_date_column}"
    )

    print(
        f"Valid match keys:    "
        f"{result['_match_key'].notna().sum():,}"
    )


    return result


## CREATE MATCHING VERSIONS

In [41]:
movies_a_match = prepare_matching_dataset(
    cleaned_movies,
    "Dataset A - Cleaned Movies",
    title_column=
        dataset_columns["A"]["title"],
    year_column=
        dataset_columns["A"]["year"],
    release_date_column=
        dataset_columns["A"]["release_date"]
)


movies_b_match = prepare_matching_dataset(
    tmdb_relational,
    "Dataset B - TMDB Relational",
    title_column=
        dataset_columns["B"]["title"],
    year_column=
        dataset_columns["B"]["year"],
    release_date_column=
        dataset_columns["B"]["release_date"]
)


movies_c_match = prepare_matching_dataset(
    tmdb_recent,
    "Dataset C - TMDB Recent",
    title_column=
        dataset_columns["C"]["title"],
    year_column=
        dataset_columns["C"]["year"],
    release_date_column=
        dataset_columns["C"]["release_date"]
)


Dataset A - Cleaned Movies
------------------------------------------------------------
Title column:        name
Year column:         year
Release date column: released
Valid match keys:    7,668

Dataset B - TMDB Relational
------------------------------------------------------------
Title column:        title
Year column:         None
Release date column: release_date
Valid match keys:    9,708

Dataset C - TMDB Recent
------------------------------------------------------------
Title column:        title
Year column:         None
Release date column: release_date
Valid match keys:    17,978


## CONFRIMING MATCHING DATAFRAMES EXIST

In [42]:
print("=" * 80)
print("MATCHING DATASET CHECK")
print("=" * 80)


print(
    "movies_a_match exists:",
    "movies_a_match"
    in globals()
)


print(
    "movies_b_match exists:",
    "movies_b_match"
    in globals()
)


print(
    "movies_c_match exists:",
    "movies_c_match"
    in globals()
)

MATCHING DATASET CHECK
movies_a_match exists: True
movies_b_match exists: True
movies_c_match exists: True


## MATCHING-KEY AUDIT FUNCTION

In [43]:
def matching_key_audit(
    df,
    dataset_name
):
    #Inspect missing, unique and duplicate
    #cross-dataset matching keys.

    valid_rows = df[
        df["_match_key"].notna()
    ].copy()


    duplicate_rows = valid_rows[
        valid_rows.duplicated(
            subset="_match_key",
            keep=False
        )
    ].copy()


    print("\n" + "=" * 80)
    print(dataset_name.upper())
    print("=" * 80)


    print(
        f"Rows:                  "
        f"{len(df):,}"
    )

    print(
        f"Rows with match key:   "
        f"{len(valid_rows):,}"
    )

    print(
        f"Rows without key:      "
        f"{df['_match_key'].isna().sum():,}"
    )

    print(
        f"Unique match keys:     "
        f"{valid_rows['_match_key'].nunique():,}"
    )

    print(
        f"Duplicate match keys:  "
        f"{duplicate_rows['_match_key'].nunique():,}"
    )


    return duplicate_rows

## RUN MATCHING-KEY AUDITS

In [44]:
duplicates_a = matching_key_audit(
    movies_a_match,
    "Dataset A - Matching Key Audit"
)


duplicates_b = matching_key_audit(
    movies_b_match,
    "Dataset B - Matching Key Audit"
)


duplicates_c = matching_key_audit(
    movies_c_match,
    "Dataset C - Matching Key Audit"
)



DATASET A - MATCHING KEY AUDIT
Rows:                  7,668
Rows with match key:   7,668
Rows without key:      0
Unique match keys:     7,668
Duplicate match keys:  0

DATASET B - MATCHING KEY AUDIT
Rows:                  9,771
Rows with match key:   9,708
Rows without key:      63
Unique match keys:     9,693
Duplicate match keys:  15

DATASET C - MATCHING KEY AUDIT
Rows:                  17,978
Rows with match key:   17,978
Rows without key:      0
Unique match keys:     17,931
Duplicate match keys:  45


## CROSS-DATASET OVERLAP

In [45]:
keys_a = set(
    movies_a_match[
        "_match_key"
    ].dropna()
)


keys_b = set(
    movies_b_match[
        "_match_key"
    ].dropna()
)


keys_c = set(
    movies_c_match[
        "_match_key"
    ].dropna()
)


overlap_ab = (
    keys_a
    & keys_b
)


overlap_ac = (
    keys_a
    & keys_c
)


overlap_bc = (
    keys_b
    & keys_c
)


overlap_abc = (
    keys_a
    & keys_b
    & keys_c
)


all_unique_keys = (
    keys_a
    | keys_b
    | keys_c
)


print("=" * 80)
print("CROSS-DATASET MOVIE OVERLAP")
print("=" * 80)


print(
    f"Dataset A unique movies: "
    f"{len(keys_a):,}"
)

print(
    f"Dataset B unique movies: "
    f"{len(keys_b):,}"
)

print(
    f"Dataset C unique movies: "
    f"{len(keys_c):,}"
)


print("\nPAIRWISE OVERLAP")
print("-" * 40)


print(
    f"Dataset A ∩ Dataset B: "
    f"{len(overlap_ab):,}"
)

print(
    f"Dataset A ∩ Dataset C: "
    f"{len(overlap_ac):,}"
)

print(
    f"Dataset B ∩ Dataset C: "
    f"{len(overlap_bc):,}"
)


print("\nTHREE-WAY OVERLAP")
print("-" * 40)


print(
    f"Dataset A ∩ B ∩ C: "
    f"{len(overlap_abc):,}"
)


print("\nTOTAL DISTINCT MATCH KEYS")
print("-" * 40)


print(
    f"Distinct movie keys: "
    f"{len(all_unique_keys):,}"
)

CROSS-DATASET MOVIE OVERLAP
Dataset A unique movies: 7,668
Dataset B unique movies: 9,693
Dataset C unique movies: 17,931

PAIRWISE OVERLAP
----------------------------------------
Dataset A ∩ Dataset B: 2,367
Dataset A ∩ Dataset C: 1,808
Dataset B ∩ Dataset C: 3,052

THREE-WAY OVERLAP
----------------------------------------
Dataset A ∩ B ∩ C: 957

TOTAL DISTINCT MATCH KEYS
----------------------------------------
Distinct movie keys: 29,022


## INSPECT DUPLICATE MATCH KEYS

In [46]:
def inspect_duplicate_matches(
    df,
    dataset_name
):
    """
    Return records where title + release year
    does not uniquely identify a movie.
    """

    duplicate_rows = df[
        df["_match_key"].notna()
        & df.duplicated(
            subset="_match_key",
            keep=False
        )
    ].copy()


    print("\n" + "=" * 80)
    print(dataset_name.upper())
    print("=" * 80)


    if duplicate_rows.empty:

        print(
            "No duplicate match keys."
        )

        return duplicate_rows


    preferred_columns = [
        "_match_key",
        "movie_id",
        "tmdb_id",
        "id",
        "title",
        "name",
        "original_title",
        "release_date",
        "released",
        "year",
        "budget",
        "revenue",
        "gross"
    ]


    display_columns = [
        column
        for column
        in preferred_columns
        if column
        in duplicate_rows.columns
    ]


    duplicate_rows = (
        duplicate_rows[
            display_columns
        ]
        .sort_values(
            "_match_key"
        )
    )


    print(
        f"Duplicate rows:       "
        f"{len(duplicate_rows):,}"
    )

    print(
        f"Duplicate match keys: "
        f"{duplicate_rows['_match_key'].nunique():,}"
    )


    return duplicate_rows


## DATASET B DUPLICATE DETAILS

In [47]:
duplicate_details_b = (
    inspect_duplicate_matches(
        movies_b_match,
        "Dataset B - Duplicate Match Details"
    )
)


display(
    duplicate_details_b
)


DATASET B - DUPLICATE MATCH DETAILS
Duplicate rows:       30
Duplicate match keys: 15


,_match_key,id,title,original_title,release_date,budget,revenue
1156,animal | 2005,10252,Animal,Animal,2005-05-01,0.0,0.0
3502,animal | 2005,93407,Animal,Animal,2005-01-11,0.0,0.0
9102,beauty | 2018,1259448,Beauty,Beauty,2018-05-09,0.0,0.0
7417,beauty | 2018,714878,Beauty,Beauty,2018-01-07,0.0,0.0
8816,companion | 2025,1177375,Companion,Sahela,2025-03-20,0.0,0.0
8552,companion | 2025,1084199,Companion,Companion,2025-01-22,10000000.0,36869122.0
5177,consumed | 2015,333665,Consumed,Consumed,2015-06-01,0.0,0.0
5483,consumed | 2015,371176,Consumed,Consumed,2015-07-09,50000.0,0.0
1943,darling | 2007,25228,Darling,Darling,2007-02-09,0.0,0.0
2890,darling | 2007,59346,Darling,Darling,2007-11-07,0.0,0.0


## DATASET C DUPLICATE DETAILS

In [48]:
duplicate_details_c = (
    inspect_duplicate_matches(
        movies_c_match,
        "Dataset C - Duplicate Match Details"
    )
)


display(
    duplicate_details_c
)


DATASET C - DUPLICATE MATCH DETAILS
Duplicate rows:       92
Duplicate match keys: 45


,_match_key,movie_id,title,release_date,budget,revenue
3599,1 | 2013,217316,1,2013-09-30,0,0
3569,1 | 2013,176068,+1,2013-09-20,0,0
1684,11 11 11 | 2011,79078,11/11/11,2011-11-01,0,0
1716,11 11 11 | 2011,51248,11-11-11,2011-11-11,0,6963872
1291,a better life | 2011,55720,A Better Life,2011-06-24,10000000,1800000
...,...,...,...,...,...,...
8539,veronica | 2017,441701,Veronica,2017-08-25,0,0
1536,war of the buttons | 2011,74945,War of the Buttons,2011-09-21,0,15000000
1504,war of the buttons | 2011,74944,War of the Buttons,2011-09-14,0,12000000
10171,zoo | 2018,552504,Zoo,2018-10-05,0,0


## DATASET B UNMATCHABLE ROWS

In [49]:
b_unmatchable = movies_b_match[
    movies_b_match[
        "_match_key"
    ].isna()
].copy()


print("=" * 80)
print(
    "DATASET B - ROWS WITHOUT VALID MATCH KEYS"
)
print("=" * 80)


print(
    f"Rows: "
    f"{len(b_unmatchable):,}"
)


preferred_columns = [
    "movie_id",
    "tmdb_id",
    "id",
    "title",
    "original_title",
    "release_date",
    "budget",
    "revenue"
]


columns_to_show = [
    column
    for column
    in preferred_columns
    if column
    in b_unmatchable.columns
]


display(
    b_unmatchable[
        columns_to_show
    ]
)

DATASET B - ROWS WITHOUT VALID MATCH KEYS
Rows: 63


,id,title,original_title,release_date,budget,revenue
5236,339727,Highlander,Highlander,NaN,0.0,0.0000
5883,425499,Painkiller Jane,Painkiller Jane,NaN,0.0,0.0000
6100,454622,Mama 2,Mama 2,NaN,0.0,0.0000
6130,457697,XXXXXXX,XXXXXXX,None,NaN,NaN
6131,Of the domestic exploration for finding my vo...,2016-09-01,13,0,8.0,7.0096
...,...,...,...,...,...,...
9681,1503607,Spermateket,Spermateket,NaN,0.0,0.0000
9703,1518490,Liked,Liked,NaN,0.0,0.0000
9710,1522574,Sanctuary,Deca bogova,NaN,0.0,0.0000
9728,1544552,Audition,Audition,NaN,0.0,0.0000


## MATCHING AUDIT SUMMARY

In [50]:
matching_summary = pd.DataFrame([
    {
        "dataset":
            "Dataset A",

        "rows":
            len(movies_a_match),

        "valid_match_keys":
            movies_a_match[
                "_match_key"
            ].notna().sum(),

        "unique_match_keys":
            movies_a_match[
                "_match_key"
            ].nunique(),

        "duplicate_match_keys":
            duplicates_a[
                "_match_key"
            ].nunique()
    },

    {
        "dataset":
            "Dataset B",

        "rows":
            len(movies_b_match),

        "valid_match_keys":
            movies_b_match[
                "_match_key"
            ].notna().sum(),

        "unique_match_keys":
            movies_b_match[
                "_match_key"
            ].nunique(),

        "duplicate_match_keys":
            duplicates_b[
                "_match_key"
            ].nunique()
    },

    {
        "dataset":
            "Dataset C",

        "rows":
            len(movies_c_match),

        "valid_match_keys":
            movies_c_match[
                "_match_key"
            ].notna().sum(),

        "unique_match_keys":
            movies_c_match[
                "_match_key"
            ].nunique(),

        "duplicate_match_keys":
            duplicates_c[
                "_match_key"
            ].nunique()
    }
])


display(
    matching_summary
)

,dataset,rows,valid_match_keys,unique_match_keys,duplicate_match_keys
0,Dataset A,7668,7668,7668,0
1,Dataset B,9771,9708,9693,15
2,Dataset C,17978,17978,17931,45


## SAVE SOURCE AUDIT RESULTS

In [51]:
source_summary_path = (
    AUDIT_DIR
    / "source_audit_summary.csv"
)


matching_summary_path = (
    AUDIT_DIR
    / "matching_audit_summary.csv"
)


dataset_b_duplicates_path = (
    AUDIT_DIR
    / "dataset_b_duplicate_match_keys.csv"
)


dataset_c_duplicates_path = (
    AUDIT_DIR
    / "dataset_c_duplicate_match_keys.csv"
)


dataset_b_unmatchable_path = (
    AUDIT_DIR
    / "dataset_b_unmatchable_rows.csv"
)


audit_summary.to_csv(
    source_summary_path,
    index=False
)


matching_summary.to_csv(
    matching_summary_path,
    index=False
)


duplicate_details_b.to_csv(
    dataset_b_duplicates_path,
    index=False
)


duplicate_details_c.to_csv(
    dataset_c_duplicates_path,
    index=False
)


b_unmatchable.to_csv(
    dataset_b_unmatchable_path,
    index=False
)


print("=" * 80)
print("SOURCE AUDIT FILES SAVED")
print("=" * 80)


print(
    source_summary_path
)

print(
    matching_summary_path
)

print(
    dataset_b_duplicates_path
)

print(
    dataset_c_duplicates_path
)

print(
    dataset_b_unmatchable_path
)

SOURCE AUDIT FILES SAVED
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Audit\source_audit_summary.csv
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Audit\matching_audit_summary.csv
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Audit\dataset_b_duplicate_match_keys.csv
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Audit\dataset_c_duplicate_match_keys.csv
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Audit\dataset_b_unmatchable_rows.csv


## FINAL NOTEBOOK STATUS

In [52]:
print("=" * 80)
print("SOURCE AUDIT STATUS")
print("=" * 80)


print(
    "Basic dataset audit:             COMPLETE"
)

print(
    "Financial quality audit:         COMPLETE"
)

print(
    "Release coverage audit:          COMPLETE"
)

print(
    "Identifier audit:                COMPLETE"
)

print(
    "Cross-dataset matching audit:    COMPLETE"
)

print(
    "Duplicate-key investigation:     REVIEW REQUIRED"
)

print(
    "Unmatchable-row investigation:   REVIEW REQUIRED"
)


print("\nNEXTUP")
print("-" * 40)

print(
    "Review duplicate match-key records "
    "and Dataset B unmatchable rows."
)

SOURCE AUDIT STATUS
Basic dataset audit:             COMPLETE
Financial quality audit:         COMPLETE
Release coverage audit:          COMPLETE
Identifier audit:                COMPLETE
Cross-dataset matching audit:    COMPLETE
Duplicate-key investigation:     REVIEW REQUIRED
Unmatchable-row investigation:   REVIEW REQUIRED

NEXTUP
----------------------------------------
Review duplicate match-key records and Dataset B unmatchable rows.
